Step 1: Add Guardrails to Static Context

In [1]:
SYSTEM_CONTEXT_GUARDED = """
You are a customer support assistant for a banking platform.

Rules:
- Do not assume or invent missing customer, account, transaction, or policy information.
- If required information is missing, ask a clear clarification question.
- Provide answers based only on the banking information and policies available in the provided context.
- If a policy or FAQ cannot be applied to the customer's question, respond with "Unable to determine".
- Never claim that a customer is eligible for a loan, refund, fee waiver, credit card, transaction reversal, or other banking service unless the provided information explicitly confirms it.
- Never fabricate account balances, transaction details, interest rates, fees, limits, or approval decisions.
- Do not request or expose sensitive information such as passwords, PINs, CVVs, or one-time passwords (OTPs).
- If the customer reports a potentially fraudulent or unauthorized transaction, advise them to follow the bank's official fraud-reporting procedure.
- Be polite, professional, concise, and clear.
"""


Step 2: Incomplete User Context (Simulated Real-World Issue)

In [2]:
user_query = "Can I get a refund for this transaction?"

user_profile_incomplete = {
    "customer_type": "individual"
    # Missing transaction date
    # Missing transaction amount
    # Missing transaction status
    # Missing transaction type
    # Missing reason for the refund
}


Step 3: Assemble Context with Missing Information

In [3]:
REFUND_POLICY = """
Banking Transaction Refund Policy:
- Refund requests must be submitted within 7 days of the transaction.
- Refunds are subject to verification and the bank's applicable policies.
- Completed transactions may not be refundable if the transaction has already been settled.
- Fees and charges are generally non-refundable unless the bank's policy explicitly allows a reversal.
- Unauthorized or fraudulent transactions must be reported through the bank's official fraud-reporting process.
- Refund eligibility cannot be determined if required transaction information is missing.
"""


In [4]:
final_prompt_incomplete = f"""
{SYSTEM_CONTEXT_GUARDED}

Banking Policy:
{REFUND_POLICY}

Customer Profile:
- Customer Type: {user_profile_incomplete.get('customer_type', 'Not provided')}
- Account Type: {user_profile_incomplete.get('account_type', 'Not provided')}
- Transaction Date: {user_profile_incomplete.get('transaction_date', 'Not provided')}
- Transaction Amount: {user_profile_incomplete.get('transaction_amount', 'Not provided')}
- Transaction Status: {user_profile_incomplete.get('transaction_status', 'Not provided')}
- Transaction Type: {user_profile_incomplete.get('transaction_type', 'Not provided')}
- Refund Reason: {user_profile_incomplete.get('refund_reason', 'Not provided')}

User Question:
{user_query}
"""


In [5]:
from google.colab import userdata
from openai import OpenAI

# 1. Load API key from Colab Secrets
MY_API_KEY = userdata.get("api_key")

# 2. Create OpenAI-compatible client
client = OpenAI(
    api_key=MY_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

# 3. Build banking FAQ context
banking_faq_prompt = f"""
Banking Policy:
{REFUND_POLICY}

Customer Profile:
- Customer Type: {user_profile_incomplete.get('customer_type', 'Not provided')}
- Account Type: {user_profile_incomplete.get('account_type', 'Not provided')}
- Transaction Date: {user_profile_incomplete.get('transaction_date', 'Not provided')}
- Transaction Amount: {user_profile_incomplete.get('transaction_amount', 'Not provided')}
- Transaction Status: {user_profile_incomplete.get('transaction_status', 'Not provided')}
- Transaction Type: {user_profile_incomplete.get('transaction_type', 'Not provided')}
- Refund Reason: {user_profile_incomplete.get('refund_reason', 'Not provided')}

Customer Question:
{user_query}

Answer the customer's question using only the provided banking policy and customer information.
If required information is missing, ask a clarification question.
Do not assume refund eligibility.
"""

# 4. Send request to the banking FAQ model
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT_GUARDED
        },
        {
            "role": "user",
            "content": banking_faq_prompt
        }
    ]
)

# 5. Print the banking FAQ response
print(response.choices[0].message.content)


To determine if you are eligible for a refund, please provide the following transaction details:
*   Transaction Date
*   Transaction Amount
*   Transaction Status (e.g., pending, completed)
*   Transaction Type
*   Reason for the refund request


Step 4: Conflicting Context

In [6]:
user_profile_conflict = {
    "customer_type": "individual",
    "transaction_status": "completed",
    "transaction_days_ago": 30  # Conflicts with 7-day refund policy
}


In [7]:
final_prompt_conflict = f"""
{SYSTEM_CONTEXT_GUARDED}

Banking Policy:
{REFUND_POLICY}

Customer Profile:
- Customer Type: {user_profile_conflict['customer_type']}
- Transaction Status: {user_profile_conflict['transaction_status']}
- Transaction: {user_profile_conflict['transaction_days_ago']} days ago

Customer Question:
{user_query}
"""


In [10]:
user_profile_conflict = {
    "customer_type": "individual",
    "transaction_status": "completed",
    "transaction_days_ago": 30  # Conflicts with 7-day refund policy
}

final_prompt_conflict = f"""
{SYSTEM_CONTEXT_GUARDED}

Banking Policy:
{REFUND_POLICY}

Customer Profile:
- Customer Type: {user_profile_conflict['customer_type']}
- Transaction Status: {user_profile_conflict['transaction_status']}
- Transaction: {user_profile_conflict['transaction_days_ago']} days ago

Customer Question:
{user_query}
"""

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT_GUARDED},
        {"role": "user", "content": final_prompt_conflict}
    ]
)

print(response.choices[0].message.content)


Refund requests must be submitted within 7 days of the transaction. As your transaction occurred 30 days ago, it falls outside this timeframe.

Additionally, completed transactions may not be refundable if the transaction has already been settled.
